In [ ]:
"""
=============================================================
 NOTEBOOK 5 — Camera System Analysis: MIJ vs QFOM
=============================================================
This notebook evaluates the predictive value of two objective
camera grading systems — MIJ and QFOM — against subjective
grading and eating quality outcomes.

Sections:
  1. Data setup — normalise panellist scores, build primal subsets
  2. Predictive power — Pearson r between camera features and
     Tenderness, Flavour, Juicy across both systems
  3. Inter-system agreement — how well MIJ and QFOM agree
     on marbling, meat colour, and fat colour
  4. Performance by primal cut — cut-specific predictive power
  5. CatBoost head-to-head — model each system separately
     and compare R² per primal

Inputs:
  - Merged panellist-level dataset with camera measurements
    (output of 01_data_merge / 02b_bias_correction)

Outputs:
  - Predictive power summary CSV
  - Inter-system agreement summary
  - Per-primal model comparison table
=============================================================
"""

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## 1. Configuration & Feature Definitions

In [ ]:
path = 'C/'

TARGETS  = ['Tenderness', 'Flavour', 'Juicy']
PRIMALS  = ['LD', 'SM', 'FT', 'GM']
MIN_N    = 50  # minimum observations required to report a correlation

MIJ_FEATURES = {
    'Marbling (Digital)': 'Objective - MIJ - Digital Marbling Score',
    'Marbling %':         'Objective - MIJ -  Marbling Percentage',
    'AUS Marbling':       'Objective - MIJ -  AUS Meat Marbling (0-9+)',
    'MSA Marbling':       'Objective - MIJ - MSA Marbling Score',
    'Meat Colour':        'Objective - MIJ -  Digital Meat Colour (AUS Meat 0 -7)',
    'Fat Colour':         'Objective - MIJ - Digital Fat Colour',
    'Fineness Index':     'Objective - MIJ - New Fineness Index \u2013 2 (range 0-11)',
    'Muscle Area':        'Objective - MIJ -  Muscle Area',
}

QFOM_FEATURES = {
    'AUS Marbling':    'Objective - QFOM -  AUS Meat Marbling',
    'MSA Marbling':    'Objective - QFOM MSA Marbling Score',
    'Meat Colour':     'Objective - QFOM -  Meat Colour',
    'Fat Colour':      'Objective - QFOM -  Fat Colour',
    'IMF%':            'Objective - QFOM -  IMF%',
    'Eye Muscle Area': 'Objective - Objective - QFOM -  Eye Muscle Area (cm2)',
    'Rib Fat (SubQ)':  'Objective - QFOM -  Rib Fat (mm) - SubQ',
}

SUBJECTIVE_MARBLING = 'Subjective - MSA Marbling Score'

AGREEMENT_PAIRS = [
    ('Marbling – Digital MIJ vs AUS QFOM',
     'Objective - MIJ - Digital Marbling Score',
     'Objective - QFOM -  AUS Meat Marbling'),
    ('Marbling – AUS MIJ vs AUS QFOM',
     'Objective - MIJ -  AUS Meat Marbling (0-9+)',
     'Objective - QFOM -  AUS Meat Marbling'),
    ('Meat Colour – MIJ vs QFOM',
     'Objective - MIJ -  Digital Meat Colour (AUS Meat 0 -7)',
     'Objective - QFOM -  Meat Colour'),
    ('Fat Colour – MIJ vs QFOM',
     'Objective - MIJ - Digital Fat Colour',
     'Objective - QFOM -  Fat Colour'),
]

## 2. Load Data & Build Normalised Targets

In [ ]:
# Load merged dataset — update path
input_file = f"{path}New_merged_meat_1_0.csv"
df_raw = pd.read_csv(input_file, low_memory=False)
print(f"Loaded: {df_raw.shape}")

# Coerce camera and target columns to numeric
all_cam_cols = list(MIJ_FEATURES.values()) + list(QFOM_FEATURES.values()) + [SUBJECTIVE_MARBLING]
for col in all_cam_cols + TARGETS:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

In [ ]:
# Filter to active graders (>=50 sessions) and build z-score normalised targets
MIN_OBS = 50
grader_counts  = df_raw.groupby('Username')[TARGETS[0]].count()
active_graders = grader_counts[grader_counts >= MIN_OBS].index
df = df_raw[df_raw['Username'].isin(active_graders)].copy()
print(f"Active grader rows: {len(df):,}")

NORM_TARGETS = []
for t in TARGETS:
    norm_col = f"{t}_norm"
    grader_stats = df.groupby('Username')[t].agg(['mean', 'std'])
    def zscore_row(row, t=t, gs=grader_stats):
        u = row['Username']
        if u not in gs.index: return np.nan
        sd = gs.loc[u, 'std']
        if sd == 0 or pd.isna(sd): return np.nan
        return (row[t] - gs.loc[u, 'mean']) / sd
    df[norm_col] = df.apply(zscore_row, axis=1)
    NORM_TARGETS.append(norm_col)
print(f"Normalised targets added: {NORM_TARGETS}")

## 3. Predictive Power — Camera Features vs Eating Quality

In [ ]:
def pearson_r(x, y):
    x = pd.to_numeric(x, errors='coerce')
    y = pd.to_numeric(y, errors='coerce')
    mask = x.notna() & y.notna()
    n = mask.sum()
    if n < MIN_N:
        return float('nan'), float('nan'), n
    r, p = stats.pearsonr(x[mask], y[mask])
    return r, p, n

def sig_stars(p):
    if np.isnan(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

groups = [
    ('MIJ',        MIJ_FEATURES),
    ('QFOM',       QFOM_FEATURES),
    ('Subjective', {SUBJECTIVE_MARBLING.replace('Subjective - ', ''): SUBJECTIVE_MARBLING}),
]

print(f"{'Feature':<32} {'System':<10}" + "".join(f"{t:>14}" for t in TARGETS) + f"{'n':>8}")
print("-" * 80)

results = []
for system, features in groups:
    print(f"[{system}]")
    for label, col in features.items():
        if col not in df.columns:
            continue
        row = {'feature': label, 'system': system, 'column': col}
        corr_str, n_vals = '', []
        for target in TARGETS:
            if target not in df.columns:
                corr_str += f"{'—':>14}"
                continue
            r, p, n = pearson_r(df[col], df[target])
            row[f'{target}_r'] = r
            n_vals.append(n)
            if np.isnan(r):
                corr_str += f"{'—':>14}"
            else:
                corr_str += f"{r:>+10.3f}{sig_stars(p):>4}"
        row['n'] = min(n_vals) if n_vals else 0
        results.append(row)
        print(f"  {label:<30} {system:<10}{corr_str}{row['n']:>8,}")

results_df = pd.DataFrame(results)
results_df.to_csv('camera_predictive_power.csv', index=False)
print("\nSaved: camera_predictive_power.csv")

## 4. Inter-System Agreement: MIJ vs QFOM

In [ ]:
mij_cols  = [c for c in MIJ_FEATURES.values()  if c in df.columns]
qfom_cols = [c for c in QFOM_FEATURES.values() if c in df.columns]
both_mask = df[mij_cols].notna().any(axis=1) & df[qfom_cols].notna().any(axis=1)
both_df   = df[both_mask].copy()

print(f"Animals with MIJ data:     {df[mij_cols].notna().any(axis=1).sum():,}")
print(f"Animals with QFOM data:    {df[qfom_cols].notna().any(axis=1).sum():,}")
print(f"Animals with BOTH systems: {both_mask.sum():,}")
print()

def agreement_label(r):
    if np.isnan(r): return '—'
    if abs(r) >= 0.70: return 'Strong'
    if abs(r) >= 0.40: return 'Moderate'
    return 'Weak'

print(f"{'Measurement Pair':<45} {'r':>8} {'p':>10} {'Sig':>5} {'Agreement':>12} {'n':>7}")
print("-" * 90)

agreement_rows = []
for label, col_a, col_b in AGREEMENT_PAIRS:
    if col_a not in df.columns or col_b not in df.columns:
        continue
    r, p, n = pearson_r(both_df[col_a], both_df[col_b])
    r_str = f"{r:+.3f}" if not np.isnan(r) else '—'
    p_str = f"{p:.4f}" if not np.isnan(p) else '—'
    print(f"  {label:<43} {r_str:>8} {p_str:>10} {sig_stars(p):>5} {agreement_label(r):>12} {n:>7,}")
    agreement_rows.append({'pair': label, 'r': r, 'p': p, 'n': n, 'agreement': agreement_label(r)})

pd.DataFrame(agreement_rows).to_csv('camera_agreement.csv', index=False)
print("\nSaved: camera_agreement.csv")

## 5. Predictive Power by Primal Cut

In [ ]:
mij_marbling_col  = 'Objective - MIJ - Digital Marbling Score'
qfom_marbling_col = 'Objective - QFOM -  AUS Meat Marbling'

by_primal_rows = []
for target in TARGETS:
    print(f"\nTarget: {target}")
    print(f"  {'Primal':<10} {'n':>6}  {'MIJ Marbling':>14}  {'QFOM Marbling':>14}  {'Subj Marbling':>14}")
    print("  " + "-" * 60)
    for primal in PRIMALS:
        sub = df[df['Primal'] == primal]
        row = {'target': target, 'primal': primal, 'n': len(sub)}
        line = f"  {primal:<10} {len(sub):>6}"
        for name, col in [('MIJ', mij_marbling_col),
                           ('QFOM', qfom_marbling_col),
                           ('Subj', SUBJECTIVE_MARBLING)]:
            if col not in df.columns:
                line += f"{'—':>16}"
                continue
            r, p, n = pearson_r(sub[col], sub[target])
            row[f'{name}_r'] = r
            if np.isnan(r):
                line += f"{'—':>16}"
            else:
                line += f"{r:>+12.3f}{sig_stars(p):>4}"
        by_primal_rows.append(row)
        print(line)

by_primal_df = pd.DataFrame(by_primal_rows)
by_primal_df.to_csv('camera_by_primal.csv', index=False)
print("\nSaved: camera_by_primal.csv")

## 6. Heatmap — Tenderness by Category × Production System

In [ ]:
df_interact = df[
    (df['Category'].notna()) & (df['Category'] != 'nan') &
    (df['Production system'].notna()) & (df['Production system'] != 'nan')
].copy()

pivot_data = df_interact.groupby(['Category', 'Production system'])['Tenderness'].mean().unstack()

plt.figure(figsize=(12, 6))
sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='RdYlGn',
            cbar_kws={'label': 'Mean Tenderness'}, vmin=5.0, vmax=7.5)
plt.title('Tenderness Heatmap: Category × Production System',
          fontweight='bold', fontsize=14)
plt.xlabel('Production System', fontsize=11)
plt.ylabel('Animal Category', fontsize=11)
plt.tight_layout()
plt.savefig('category_production_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. CatBoost Head-to-Head: MIJ vs QFOM vs Subjective
Build one model per system per primal and compare R² directly.

In [ ]:
SUBJECTIVE_FEATURES = [
    'Subjective  -MSA Ossification', 'Subjective - MSA Eye Muscle Area (cm2)',
    'Subjective - MSA Total Rib Fat (mm)', 'Subjective - MSA Subcutaneous Rib Fat (mm)',
    'Subjective - MSA Hump Height (mm)', 'Subjective - AUS MEAT Marbling Score',
    'Subjective - MSA Marbling Score', 'Subjective - MSA Lean Colour Score',
    'Subjective-MSA Fat Colour',
]

QFOM_ANCHOR = [
    'Objective - Objective - QFOM -  Eye Muscle Area (cm2)',
    'Objective - QFOM -  Rib Fat (mm) - SubQ',
    'Objective - QFOM -  AUS Meat Marbling',
    'Objective - QFOM -  Meat Colour',
    'Objective - QFOM -  Fat Colour',
]
MIJ_ANCHOR = [
    'Objective - MIJ -  Marbling Percentage',
    'Objective - MIJ - Digital Marbling Score',
    'Objective - MIJ - Digital Fat Colour',
]

MODEL_PARAMS = {
    'iterations': 500, 'learning_rate': 0.05, 'depth': 4,
    'l2_leaf_reg': 10, 'random_seed': 42, 'loss_function': 'RMSE',
    'verbose': 0,
}

TARGET = 'Tenderness_norm'

comparison_rows = []

for primal in PRIMALS:
    sub = df[df['Primal'] == primal].copy()
    sub = sub.dropna(subset=[TARGET])
    if len(sub) < 100:
        continue

    for system_name, feature_list, anchor_cols in [
        ('QFOM + Subj', list(QFOM_FEATURES.values()) + SUBJECTIVE_FEATURES, QFOM_ANCHOR),
        ('MIJ + Subj',  list(MIJ_FEATURES.values())  + SUBJECTIVE_FEATURES, MIJ_ANCHOR),
        ('Subj only',   SUBJECTIVE_FEATURES, SUBJECTIVE_FEATURES[:3]),
    ]:
        avail_anchor = [c for c in anchor_cols if c in sub.columns]
        if not avail_anchor:
            continue
        camera_mask = sub[avail_anchor].notna().all(axis=1)
        sub_cam = sub[camera_mask].copy()
        if len(sub_cam) < 50:
            continue

        feats = [c for c in feature_list if c in sub_cam.columns]
        X = sub_cam[feats].copy()
        y = sub_cam[TARGET].values
        cat_feats = [c for c in feats if X[c].dtype == 'object']
        for c in cat_feats:
            X[c] = X[c].astype(str).fillna('missing')
        cat_idx = [X.columns.get_loc(c) for c in cat_feats]

        kf = GroupKFold(n_splits=5)
        group_col_avail = 'steak_id' if 'steak_id' in sub_cam.columns else None
        groups = sub_cam[group_col_avail].values if group_col_avail else np.arange(len(sub_cam))

        fold_r2s = []
        for tr_idx, te_idx in kf.split(X, y, groups):
            X_tr, X_te = X.iloc[tr_idx].copy(), X.iloc[te_idx].copy()
            y_tr, y_te = y[tr_idx], y[te_idx]
            model = CatBoostRegressor(**MODEL_PARAMS)
            model.fit(Pool(X_tr, y_tr, cat_features=cat_idx),
                      eval_set=Pool(X_te, y_te, cat_features=cat_idx))
            preds = model.predict(X_te)
            ss_res = np.sum((y_te - preds) ** 2)
            ss_tot = np.sum((y_te - y_te.mean()) ** 2)
            fold_r2s.append(1 - ss_res / ss_tot)

        comparison_rows.append({
            'Primal': primal, 'System': system_name,
            'n': len(sub_cam), 'R2_mean': round(np.mean(fold_r2s), 4),
            'R2_std': round(np.std(fold_r2s), 4)
        })
        print(f"{primal} | {system_name:<20} n={len(sub_cam):>5}  "
              f"R²={np.mean(fold_r2s):.4f} ± {np.std(fold_r2s):.4f}")

comparison_df = pd.DataFrame(comparison_rows)
print("\n" + comparison_df.pivot(index='Primal', columns='System', values='R2_mean').to_string())
comparison_df.to_csv('camera_model_comparison.csv', index=False)
print("\nSaved: camera_model_comparison.csv")